In [4]:
import pandas as pd
df = pd.read_csv("../results/exp_20250627_042542/results_updated.csv")
df['result']

0      {\n    "namespace": "guest",\n    "name": "res...
1      {\n    "namespace": "guest",\n    "name": "eff...
2      {\n    "namespace": "guest",\n    "name": "eff...
3      {\n    "namespace": "guest",\n    "name": "ber...
4      {\n    "namespace": "guest",\n    "name": "ber...
                             ...                        
917    {\n    "namespace": "guest",\n    "name": "ale...
918    {\n    "namespace": "guest",\n    "name": "ale...
919    {\n    "namespace": "guest",\n    "name": "ale...
920    {\n    "namespace": "guest",\n    "name": "ale...
921    {\n    "namespace": "guest",\n    "name": "dis...
Name: result, Length: 922, dtype: object

In [8]:
# import pandas as pd
# import json

def parse_json_safe(val):
    if pd.isna(val) or val == "None":
        return None
    if isinstance(val, dict):  # Already a dict
        return val
    try:
        return json.loads(val)
    except Exception:
        return None

def extract_name(res,name):
    if isinstance(res, dict):
        return res.get(name)
    return None




def extract_error_message(res):
    if isinstance(res, dict):
        resp = res.get("response", {})
        if not resp.get("success", True):
            return resp.get("result", {}).get("error")
    return None

def has_cuda_error(msg):
    if isinstance(msg, str):
        return "cuda" in msg.lower() or "cudnn" in msg.lower()
    return False

def extract_success(res):
    if isinstance(res, dict):
        return res.get("response", {}).get("success", True)
    return False



In [10]:
"""
Phase-level latency attribution for the Middleware rebuttal.

Each row of results_updated.csv is an OpenWhisk activation record (JSON).
This parses the per-phase timestamps and answers, per function:

  1. Where does NMIG's extra latency actually go? (phase-by-phase delta)
  2. O1 from Sec 3.1.1: GPU-resident time / inference time  (target ~1.0)
  3. Is the overhead constant across models, or does it scale with model size?

No new experiments -- this re-reads runs you already have.
"""

import json
import numpy as np
import pandas as pd

# ---------------- config ----------------
RESULTS_DIR = "../results"
BASELINE = "Openwhisk"
TREATMENT = "NMIG"

experiments = [
    {"id": "exp_20250626_205804", "name": "Openwhisk"},
    {"id": "exp_20250627_042542", "name": "NMIG"},
    {"id": "exp_20250626_155721", "name": "Histogram"},
    {"id": "exp_20250626_113035", "name": "Pagurus"},
]

results = {}
for exp in experiments:
    # load the CSV for this experiment
    path = f"../results/{exp['id']}/results_updated.csv"
    df = pd.read_csv(path)

    # parse JSON and compute latency
    df["result_parsed"] = df["result"].apply(parse_json_safe)
    df["name"] = df["result_parsed"].apply(lambda res: extract_name(res, "name"))
    df['name'] = df['name'].str.replace(r'_p$', '', regex=True)
    df["start"] = df["result_parsed"].apply(lambda res: extract_name(res, "start"))
    df["end"] = df["result_parsed"].apply(lambda res: extract_name(res, "end"))
    df['latency'] = (df['end'] - df['start'])/1000
    df["error_message"] = df["result_parsed"].apply(extract_error_message)
    df["has_cuda_or_cudnn_error"] = df["error_message"].apply(has_cuda_error)
    df["success"] = df["result_parsed"].apply(extract_success)
    df["error"] = df["result_parsed"].isna() | (~df["success"])
    error_counts = df['error'].value_counts()
    results[exp['name']] = {}
    results[exp['name']]['true_val'] = error_counts.get(True, 0)
    results[exp['name']]['false_val'] = error_counts.get(False, 0)
    results[exp['name']]['false_val'] = df['latency'].mean()
    df_valid = df[df["error"] != True]
    avg_latency_per_name_dict = df_valid.groupby("name")["latency"].mean().to_dict()
    print(avg_latency_per_name_dict)
    
   
    # df = df[df['latency'] < 300]

print(results)

{'alexnet': 1.2156033057851239, 'bert': 1.3614436619718309, 'distilgpt2': 2.620737226277372, 'efficientnet': 1.0782916666666666, 'googlenet': 12.588571428571429, 'inception': 1.096030303030303, 'resnet50': 0.47011250000000004}
{'alexnet': 4.270818181818182, 'bert': 3.160063291139241, 'distilgpt2': 4.987741258741258, 'efficientnet': 3.111483333333333, 'googlenet': 11.301970238095238, 'inception': 2.7369015151515153, 'resnet50': 2.445325}
{'alexnet': 1.2470097087378642, 'bert': 1.3688344370860925, 'distilgpt2': 2.9020846153846156, 'efficientnet': 1.1080583333333334, 'googlenet': 8.629614864864864, 'inception': 1.1528939393939392, 'resnet50': 0.523925}
{'alexnet': 1.2182809917355373, 'bert': 1.2818211920529803, 'distilgpt2': 2.477716312056738, 'efficientnet': 1.0842833333333333, 'googlenet': 12.103755952380952, 'inception': 1.067378787878788, 'resnet50': 0.51505}
{'Openwhisk': {'true_val': 22, 'false_val': 3.4143503253796097}, 'NMIG': {'true_val': 0, 'false_val': 4.943939262472885}, 'Hist

In [11]:
"""
Per-function, per-phase latency attribution for the Middleware rebuttal.

Re-reads runs you already have -- no new experiments.

Answers:
  1. Per-function latency, NMIG vs baselines
  2. WHERE NMIG's extra time goes (phase-by-phase delta)
  3. Is the overhead a constant, or does it scale with model size?
  4. O1 (Sec 3.1.1): GPU-resident time / inference time, target ~1.0
  5. GPU memory still allocated when the invocation ends

Note: latency here is (end - start), which EXCLUDES queueing (waitTime).
Confirm Table 2 was built the same way before quoting these numbers.
"""

import json
import re
import numpy as np
import pandas as pd

# ---------------- config ----------------
RESULTS_DIR = "../results"
BASELINE = "Openwhisk"
TREATMENT = "NMIG"

experiments = [
    {"id": "exp_20250626_205804", "name": "Openwhisk"},
    {"id": "exp_20250627_042542", "name": "NMIG"},
    {"id": "exp_20250626_155721", "name": "Histogram"},
    {"id": "exp_20250626_113035", "name": "Pagurus"},
]

# resident GPU memory per model, Sec 2.2 (MB)
RESIDENT_MB = {
    "alexnet": 841.15, "resnet50": 879.42, "efficientnet_b7": 4543.88,
    "efficientnet": 4543.88, "distilgpt2": 667.47, "bert": 919.86,
}


# ---------------- helpers ----------------
def parse_json_safe(x):
    if isinstance(x, dict):
        return x
    if not isinstance(x, str):
        return None
    try:
        return json.loads(x)
    except (json.JSONDecodeError, TypeError):
        return None


def extract_name(res, key):
    if not isinstance(res, dict):
        return None
    return res.get(key)


def extract_success(res):
    if not isinstance(res, dict):
        return False
    return bool(res.get("response", {}).get("success", False))


def extract_error_message(res):
    if not isinstance(res, dict):
        return None
    r = res.get("response", {}).get("result", {})
    if isinstance(r, dict):
        for k in ("error", "message", "errorMessage"):
            if k in r:
                return str(r[k])
    return None


def has_cuda_error(msg):
    if not isinstance(msg, str):
        return False
    return bool(re.search(r"cuda|cudnn|out of memory", msg, re.I))


def get_measurement(res):
    try:
        m = res["response"]["result"]["measurement"]
        return m if isinstance(m, dict) else {}
    except (TypeError, KeyError):
        return {}


def phase_secs(res):
    """Auto-pair every *_start with its *_end -- works for image, video, NLP."""
    m = get_measurement(res)
    out = {}
    for k in m:
        if k.endswith("_start"):
            stem = k[: -len("_start")]
            end = f"{stem}_end"
            if end in m:
                try:
                    out[f"ph_{stem}"] = (float(m[end]) - float(m[k])) / 1000.0
                except (TypeError, ValueError):
                    pass
    # O1: GPU held from first transfer until inference ends, vs active compute
    try:
        gpu = (float(m["inference_end"]) - float(m["gpu_transfer_start"])) / 1000.0
        inf = (float(m["inference_end"]) - float(m["inference_start"])) / 1000.0
        out["gpu_resident_s"] = gpu
        out["inference_s"] = inf
        out["o1_ratio"] = gpu / inf if inf > 0 else np.nan
    except (KeyError, TypeError, ValueError):
        pass
    return out


def gpu_mem_after(res):
    try:
        gpus = res["response"]["result"]["system_stats_after"]["gpus"]
        return sum(float(g.get("memory_allocated_mb", 0)) for g in gpus)
    except (TypeError, KeyError, ValueError):
        return np.nan


def annotation(res, key):
    try:
        for a in res.get("annotations", []) or []:
            if a.get("key") == key:
                return a.get("value")
    except AttributeError:
        pass
    return np.nan


# ---------------- load ----------------
results = {}

for exp in experiments:
    path = f"{RESULTS_DIR}/{exp['id']}/results_updated.csv"
    df = pd.read_csv(path)

    df["result_parsed"] = df["result"].apply(parse_json_safe)
    df["name"] = df["result_parsed"].apply(lambda r: extract_name(r, "name"))
    df["name"] = df["name"].astype(str).str.replace(r"_p$", "", regex=True).str.lower()
    df["start"] = df["result_parsed"].apply(lambda r: extract_name(r, "start"))
    df["end"] = df["result_parsed"].apply(lambda r: extract_name(r, "end"))
    df["latency"] = (df["end"] - df["start"]) / 1000.0
    df["wait_s"] = df["result_parsed"].apply(lambda r: annotation(r, "waitTime")) / 1000.0
    df["init_s"] = df["result_parsed"].apply(lambda r: annotation(r, "initTime")) / 1000.0
    df["error_message"] = df["result_parsed"].apply(extract_error_message)
    df["has_cuda_or_cudnn_error"] = df["error_message"].apply(has_cuda_error)
    df["success"] = df["result_parsed"].apply(extract_success)
    df["error"] = df["result_parsed"].isna() | (~df["success"])

    valid = df[~df["error"]].copy()
    ph = pd.DataFrame(list(valid["result_parsed"].apply(phase_secs)), index=valid.index)
    valid = pd.concat([valid, ph], axis=1)
    valid["gpu_mb_after"] = valid["result_parsed"].apply(gpu_mem_after)

    results[exp["name"]] = {
        "n_total": len(df),
        "n_fail": int(df["error"].sum()),
        "n_cuda_fail": int(df["has_cuda_or_cudnn_error"].sum()),
        "mean_latency": valid["latency"].mean(),
        "p95_latency": valid["latency"].quantile(0.95),
        "per_func": valid.groupby("name").mean(numeric_only=True),
        "counts": valid.groupby("name").size(),
    }

    print(f"[{exp['name']:10s}] {len(df):5d} activations, "
          f"{results[exp['name']]['n_fail']:4d} failed "
          f"({results[exp['name']]['n_cuda_fail']} CUDA), "
          f"mean={results[exp['name']]['mean_latency']:.2f}s")

# ---------------- 1. per-function latency ----------------
print("\n" + "=" * 74)
print("PER-FUNCTION MEAN LATENCY (s), completed invocations only")
print("=" * 74)
lat = pd.DataFrame({n: results[n]["per_func"]["latency"] for n in results})
print(lat.round(3).to_string())

# ---------------- 2. where the delta goes ----------------
b = results[BASELINE]["per_func"]
t = results[TREATMENT]["per_func"]
phase_cols = [c for c in t.columns if c.startswith("ph_") and c in b.columns]
cols = ["latency", "init_s"] + phase_cols
cols = [c for c in cols if c in b.columns and c in t.columns]
common = b.index.intersection(t.index)
delta = (t.loc[common, cols] - b.loc[common, cols]).round(3)

print("\n" + "=" * 74)
print(f"{TREATMENT} minus {BASELINE}: per function, per phase (s)")
print("=" * 74)
print(delta.to_string())

print("\nMean delta by phase (which phases explain the gap):")
md = delta.mean().sort_values(ascending=False)
print(md.round(3).to_string())
ph_sum = md[[c for c in phase_cols if c in md.index]].sum()
print(f"\n  total latency delta : {md.get('latency', np.nan):.3f}s")
print(f"  sum of phase deltas : {ph_sum:.3f}s")

# ---------------- 3. constant or size-dependent? ----------------
d = delta["latency"]
print(f"\nlatency delta: mean={d.mean():.2f}  median={d.median():.2f}  "
      f"sd={d.std():.2f}  min={d.min():.2f}  max={d.max():.2f}  "
      f"spread={d.max() - d.min():.2f}")

sz = pd.Series({f: RESIDENT_MB.get(f, np.nan) for f in d.index}).dropna()
if len(sz) >= 3:
    r = np.corrcoef(sz.values, d.loc[sz.index].values)[0, 1]
    print(f"corr(resident model MB, latency delta) = {r:.2f}  over {len(sz)} functions")
    if abs(r) < 0.4:
        print("  -> CONSTANT: claim 'a fixed ~Xs per-invocation cost across all functions'")
    else:
        print("  -> SCALES with model size: claim a RANGE, not a constant")

# ---------------- 4. O1 ----------------
o1 = pd.DataFrame({n: results[n]["per_func"]["o1_ratio"]
                   for n in results if "o1_ratio" in results[n]["per_func"].columns})
if not o1.empty:
    print("\n" + "=" * 74)
    print("O1 (Sec 3.1.1): GPU-resident / inference time, target ~1.0")
    print("=" * 74)
    print(o1.round(2).to_string())

# ---------------- 5. memory at invocation end ----------------
mem = pd.DataFrame({n: results[n]["per_func"]["gpu_mb_after"]
                    for n in results if "gpu_mb_after" in results[n]["per_func"].columns})
if not mem.empty:
    print("\n" + "=" * 74)
    print("GPU memory allocated at invocation end (MB)")
    print("=" * 74)
    print(mem.round(1).to_string())

# ---------------- 6. reliability ----------------
print("\n" + "=" * 74)
print("RELIABILITY")
print("=" * 74)
rel = pd.DataFrame({n: {"total": results[n]["n_total"],
                        "failed": results[n]["n_fail"],
                        "cuda_failed": results[n]["n_cuda_fail"],
                        "mean_lat": results[n]["mean_latency"],
                        "p95_lat": results[n]["p95_latency"]}
                    for n in results}).T
print(rel.round(3).to_string())

[Openwhisk ]   922 activations,   22 failed (0 CUDA), mean=3.47s
[NMIG      ]   922 activations,    0 failed (0 CUDA), mean=4.94s
[Histogram ]   903 activations,   39 failed (0 CUDA), mean=2.68s
[Pagurus   ]   921 activations,    8 failed (0 CUDA), mean=3.33s

PER-FUNCTION MEAN LATENCY (s), completed invocations only
              Openwhisk    NMIG  Histogram  Pagurus
name                                               
alexnet           1.216   4.271      1.247    1.218
bert              1.361   3.160      1.369    1.282
distilgpt2        2.621   4.988      2.902    2.478
efficientnet      1.078   3.111      1.108    1.084
googlenet        12.589  11.302      8.630   12.104
inception         1.096   2.737      1.153    1.067
resnet50          0.470   2.445      0.524    0.515

NMIG minus Openwhisk: per function, per phase (s)
              latency  init_s  ph_gpu_transfer  ph_image_load  ph_inference  ph_lib_load  ph_model_load  ph_output  ph_preprocess
name                            